# Harmonise (step 8a): cohort, clock, units and cleaning rules

Turns the concept map into long tables, one parquet per site and concept group, in `derived/harmonised/` (row-level: stays in the SDE, git-ignored). Columns: site, stay_key, concept, hour (hours since the stay's t0, true elapsed time), value, value_text (text concepts), flag, source.

Rules (see `DATA_ISSUES.md` and the `harmonise.py` docstring): stays over 6 hours only; all chart times to UTC before subtracting t0 (SMH chart times are UTC, the rest London local); haemoglobin to g/L; haematocrit to percent; placeholder zeros to missing; GOSH diastolic equal to systolic to missing; GOSH base excess derived from pH and bicarbonate and flagged `derived`; tidal volume unit stripped; I:E ratio as the x of 1:x.

Fluids and drugs are not in this notebook (see `harmonise_fluids`, next step). Only counts and hours leave the SDE.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from datetime import date
from pathlib import Path
import pandas as pd
open_picu_dir = Path("/home/jormond/Documents/DS217/users/jormond/open_picu")
sys.path.insert(0, str(open_picu_dir / "code"))
import concepts as C
import harmonise as H

In [ ]:
open_picu_dir = Path("/home/jormond/Documents/DS217/users/jormond/open_picu")
assembled_dir = open_picu_dir / "derived" / "assembled"
reports_dir = open_picu_dir / "reports"
harmonised_dir = open_picu_dir / "derived" / "harmonised"
static = pd.read_parquet(open_picu_dir / "derived" / "static" / "pooled_static.parquet")
today = date.today().isoformat()
co = H.cohort(static)
print(H.__file__)
print(co.groupby("site").size().to_dict(), "stays over 6 hours of", static.groupby("site").size().to_dict())

## 1. Is the clock right?
Median hours of the first heart rate after t0, by site and season. If the time zones are handled correctly, summer and winter should agree (an error would show as a one-hour step). The share before t0 or more than 24 h after shows how far chart times sit from the recorded admission.

In [ ]:
clock = H.time_alignment_check(assembled_dir, static)
clock

## 2. Harmonise every non-fluid, non-drug concept
Reads each source with its row times, applies the rules and writes one parquet per site and group. The report has counts only: rows read, outside the cohort, with a bad time, non-numeric, set missing, converted, derived, and the 5th, 50th and 95th percentile of `hour`. This is slow (millions of rows); run it once.

In [ ]:
report = H.harmonise_all(assembled_dir, static, harmonised_dir)
report.to_csv(Path(reports_dir) / "harmonise_report.csv", index=False)
report.drop(columns="cohort_stays")

## 2b. Fluids
Fluids have their own definitions (see the `harmonise.py` section header and `DATA_ISSUES.md` G8, S3, R6): hourly ml, with `intake_total` at all sites and `intake_iv` (infusion, colloids and drug volume; no blood, enteral or TPN) at RBH and SMH only. GOSH hourly amounts come from differencing the 24 h running totals before the Epic switch and from `caboodle_fluid_balance` after it. Nothing is removed: hours above the 99.9th percentile of ml per kg per hour (per site and concept) are flagged `outlier`, and the cap values are reported so the choice of percentile can be reviewed.

In [ ]:
fl_report, fl_caps = H.harmonise_fluids(assembled_dir, static, harmonised_dir)
fl_report.to_csv(Path(reports_dir) / "harmonise_fluids_report.csv", index=False)
fl_caps.to_csv(Path(reports_dir) / "harmonise_fluids_caps.csv", index=False)
display(fl_report[["site", "concept", "source", "stays", "rows", "rows_gap", "rows_outlier", "hour_median"]])
display(fl_caps)

## 2c. Do the harmonised rows lie inside the stay?
For every harmonised file, per site and concept: the share of rows more than 24 h before t0, from 24 h before up to t0, inside the stay (t0 to `los_hours`) and after the end of the stay. RBH blood gas and labs reach far before and after t0 (5th percentile about -1,600 h), which suggests samples from outside the stay. Nothing is removed here; this shows how much would be.

In [ ]:
window = H.window_check(harmonised_dir, static, assembled_dir=assembled_dir)
window.to_csv(Path(reports_dir) / "harmonise_window.csv", index=False)
window[(window["pct_before_24h"] > 1) | (window["pct_after_end"] > 1)].sort_values(["site", "pct_before_24h"], ascending=[True, False])

### 2d. RBH: why do the vitals run past the recorded end?
Uses the ward times in the RBH vitals table: ward in and out relative to t0, whether a stay key has several ward episodes or admission keys, and how the last heart rate compares with the static end and with the ward out. Also lists the clinical units that appear in the vitals, in case the rows after the end belong to another unit.

In [ ]:
rbh_end = H.rbh_end_check(assembled_dir, static)
print(rbh_end.to_string())
C.list_values(assembled_dir, "RBH", "ai_picu_vitals", "clinicalUnit")

**By clinical unit.** For each unit in the RBH vitals: rows, stays, and the percent of heart-rate rows more than 24 h before t0, from 24 h before to t0, inside the stay and after the static end. Shows which units the rows past the recorded end belong to.

In [ ]:
pd.set_option("display.width", 200)
H.rbh_unit_check(assembled_dir, static)

## 3. Save the results for review

In [ ]:
flagged = window[(window["pct_before_24h"] > 1) | (window["pct_after_end"] > 1)]
out = Path(reports_dir) / f"harmonise_review_{today}.md"
sections = {
    "Fluids report": fl_report,
    "Fluids caps (ml per kg per hour)": fl_caps,
    "Window check (concepts with more than 1% of rows before -24 h or after the stay)": flagged,
}
out.write_text("\n\n".join(f"## {k}\n\n```\n{v.to_string()}\n```" for k, v in sections.items()) + "\n")
print("written:", out)

In [ ]:
r = report.assign(pct_removed=(100 * report["rows_outside_window"] / (report["rows"] + report["rows_outside_window"])).round(1))
print(r.groupby("site")[["rows", "rows_outside_window"]].sum())
r[r["pct_removed"] > 0.5][["site", "concept", "rows", "rows_outside_window", "pct_removed"]]

### Where do the rows of a stay lie compared with its recorded length?
Per stay, the last row against the recorded length of stay, and the first row against t0, for heart rate, SpO2, intake, pH and sodium. A few stays with a very long tail point to overlapping episodes under one stay key. A small excess in most stays points to a length of stay that is too short.

In [ ]:
span = H.stay_span_check(harmonised_dir, static)
span.to_csv(Path(reports_dir) / "harmonise_span.csv", index=False)
span